In [1]:
from Utils.label_data_helper import *


# Methodology

Sentiment labels are assigned to each news article based on the sign of this aggregated `three-day excess return`. This excess return is calculated from the day a news article is first published and extends over the two subsequent days. To elaborate, excess return is defined as the difference between the return of a particular stock and the overall market return on the same day. This calculation is not limited to the day the news is published; instead, it aggregates the returns for the following two days as well, providing a
comprehensive three-day outlook.

A positive aggregated excess return leads to a sentiment label of `1`, indicating a positive sentiment. Conversely, a non-positive aggregated excess return results in a sentiment label of `0`, suggesting a negative sentiment.

In [2]:
stocks = pd.read_csv("Data/stocks_data.csv", header=[0, 1], index_col=0)
stocks_info = pd.read_csv("Data/SPY_companies_info.csv")
# Separate DataFrames by the top-level column (first row of headers)
stocks_dict = {key: stocks[key] for key in stocks.columns.levels[0]}
market = pd.read_csv("Data/market_data.csv")
market = market.set_index('Date')

In [3]:
# Calculate excess return sentiment label
def excess_return_sentiment_label(stock_data, market_data, num_days=3, price_used="Adj Close"):
    if price_used not in stock_data.columns or price_used not in market_data.columns:
        raise ValueError(f"Column '{price_used}' is missing in stock or market data.")

    stock_df = stock_data[[price_used]].copy()
    market_df = market_data[[price_used]].copy()
    market_df = market_df.add_prefix("SPY_")
    merged_df = stock_df.join(market_df)
    merged_df = merged_df.pct_change()


    merged_df["excess_returns"] = merged_df["Adj Close"] - merged_df["SPY_Adj Close"]
    merged_df["X_days_excess_returns"] =\
        (
            (1 + merged_df["excess_returns"])
            .rolling(num_days)
            .apply(lambda x: x.cumprod()[-1], raw=True)
            .shift(-(num_days - 1))
            - 1
        )

    # Label for 1 excess returns > 0, 0 otherwise
    merged_df["sentiment_label"] = (merged_df["X_days_excess_returns"] > 0).astype(int)
    merged_df = merged_df.reset_index()
    merged_df = merged_df[["Date", "sentiment_label"]]
    return merged_df


ticker_senti_date = pd.DataFrame()

for ticker in stocks_dict:
    result = excess_return_sentiment_label(stocks_dict[ticker], market)
    result["Ticker"] = ticker
    ticker_senti_date = pd.concat([ticker_senti_date, result], ignore_index=True)

ticker_senti_date.dropna(subset=["sentiment_label"])
ticker_senti_date

,Date,sentiment_label,Ticker
0,2014-01-02,0,A
1,2014-01-03,1,A
2,2014-01-06,1,A
3,2014-01-07,1,A
4,2014-01-08,1,A
...,...,...,...
1381736,2024-11-22,0,ZTS
1381737,2024-11-25,0,ZTS
1381738,2024-11-26,0,ZTS
1381739,2024-11-27,0,ZTS


In [4]:
# Process train and test set
news_train_df = pd.read_csv("./Data/news_data_train.csv")
news_test_df = pd.read_csv("./Data/news_data_test.csv")

news_train_combined_df = pd.merge(
    news_train_df, ticker_senti_date,
    left_on=["date", "Ticker"],
    right_on=["Date", "Ticker"],
    how="left"
)
news_train_combined_df = news_train_combined_df.drop(["date", "Ticker"], axis=1)
news_train_combined_df.dropna(subset=["Date"], inplace=True)

news_test_combined_df = pd.merge(
    news_test_df, ticker_senti_date,
    left_on=["date", "Ticker"],
    right_on=["Date", "Ticker"],
    how="left"  # Use 'left' to keep all rows from df1
)
news_test_combined_df = news_test_combined_df.drop(["date", "Ticker"], axis=1)
news_test_combined_df.dropna(subset=["Date"], inplace=True)

In [5]:
# Format
train_df = news_train_combined_df.copy()
train_df["Date"] = pd.to_datetime(train_df["Date"])
train_df['sentiment_label'] = train_df['sentiment_label'].astype(int)

test_df = news_test_combined_df.copy()
test_df["Date"] = pd.to_datetime(test_df["Date"])
test_df['sentiment_label'] = test_df['sentiment_label'].astype(int)

train_df.columns = [i.lower() for i in train_df.columns]
test_df.columns = [i.lower() for i in test_df.columns]

In [6]:
# Further split for train and val set
val_df = train_df[train_df['date'].dt.year > 2021].reset_index(drop=True)
train_df = train_df[train_df['date'].dt.year <= 2021].reset_index(drop=True)

In [7]:
train_df

,title,source,topic,company name(s) - cleaned,date,sentiment_label
0,Agilent Technologies Introduces New Version of...,Business Wire,Life Sciences Tools and Services,"Agilent Technologies, Inc.",2014-01-06,1
1,Comcast Corporation Partners with Samsung Elec...,Business Wire,Comcast Corporation (NasdaqGS:CMCSA) (Cable an...,Comcast Corporation,2014-01-06,1
2,Agilent Technologies Introduces ICP-MS and MP-...,Business Wire,Life Sciences Tools and Services,"Agilent Technologies, Inc.",2014-01-06,1
3,Agilent Technologies Inc. Introduces New Exter...,Business Wire,Life Sciences Tools and Services,"Agilent Technologies, Inc.",2014-01-08,1
4,Agilent Technologies Introduces First USB 3.0 ...,Other,Life Sciences Tools and Services,"Agilent Technologies, Inc.",2014-01-09,1
...,...,...,...,...,...,...
187155,"Zoetis Inc., Q3 2021 Earnings Call, Nov 04, 2021",Business Wire; Company Website,Pharmaceuticals,Zoetis Inc.,2021-11-04,1
187156,"Credit Suisse Group AG, 30th Annual Credit Sui...",PR Newswire; Business Wire; GlobeNewswire; Com...,"1Life Healthcare, Inc. (Health Care Services);...","1Life Healthcare, Inc.; 23andMe Holding Co.",2021-11-08,1
187157,Zoetis Inc. Presents at 30th Annual Credit Sui...,PR Newswire; Business Wire; GlobeNewswire; Com...,Pharmaceuticals,Zoetis Inc.,2021-11-09,1
187158,Zoetis Inc. Presents at 2021 HMG Live! Pacific...,GlobeNewswire; Company Website,Pharmaceuticals,Zoetis Inc.,2021-11-18,1


In [8]:
train_df.to_csv("./Data/train_labelled.csv", index=False)

In [9]:
val_df

,title,source,topic,company name(s) - cleaned,date,sentiment_label
0,"Agilent Technologies, Inc., $ 0.21, Cash Divid...",Financial Times,Life Sciences Tools and Services,"Agilent Technologies, Inc.",2022-01-03,0
1,"Agilent Technologies, Inc. Presents at Goldman...",PR Newswire; Business Wire; Company Website,Life Sciences Tools and Services,"Agilent Technologies, Inc.",2022-01-06,0
2,"Agilent Technologies, Inc. Presents at JPMorga...",PR Newswire; Business Wire; Other; GlobeNewswi...,Life Sciences Tools and Services,"Agilent Technologies, Inc.",2022-01-11,1
3,Agilent Announces the Innovative Seahorse XF P...,Business Wire,Life Sciences Tools and Services,"Agilent Technologies, Inc.",2022-01-24,0
4,"Samsung Biologics Co.,Ltd. (KOSE:A207940) reac...",Capital IQ Transaction Database,Biogen Inc. (NasdaqGS:BIIB) (Biotechnology); S...,Biogen Inc.,2022-01-27,1
...,...,...,...,...,...,...
60164,Zoetis Inc. Reports Earnings Results for the T...,S&P Capital IQ Financials Database,Pharmaceuticals,Zoetis Inc.,2023-11-02,1
60165,Zoetis Inc. Provides Earnings Guidance for the...,Business Wire,Pharmaceuticals,Zoetis Inc.,2023-11-02,1
60166,"Zoetis Inc., Q3 2023 Earnings Call, Nov 02, 2023",Business Wire,Pharmaceuticals,Zoetis Inc.,2023-11-02,1
60167,Zoetis Inc. Presents at Piper Sandler 35th Ann...,PR Newswire; Business Wire; Other; GlobeNewswi...,Pharmaceuticals,Zoetis Inc.,2023-11-28,0


In [10]:
val_df.to_csv("./Data/val_labelled.csv", index=False)

In [11]:
test_df

,title,source,topic,company name(s) - cleaned,date,sentiment_label
0,"Agilent Technologies, Inc. Presents at BIO Par...",Business Wire; GlobeNewswire; Company Website,Life Sciences Tools and Services,"Agilent Technologies, Inc.",2024-01-08,0
1,"Agilent Technologies, Inc. Presents at J.P. Mo...",PR Newswire; Business Wire; Other; GlobeNewswi...,Life Sciences Tools and Services,"Agilent Technologies, Inc.",2024-01-09,0
2,"Agilent Technologies, Inc. Announces New Prote...",Business Wire,Life Sciences Tools and Services,"Agilent Technologies, Inc.",2024-01-16,0
3,"Agilent Technologies, Inc. Presents at 23rd An...",Company Website,Life Sciences Tools and Services,"Agilent Technologies, Inc.",2024-01-16,0
4,"Agilent Technologies, Inc. - Special Call",Company Website,Life Sciences Tools and Services,"Agilent Technologies, Inc.",2024-01-17,0
...,...,...,...,...,...,...
23844,Zoetis Inc. Announces Retirement of Linda Rhod...,SEC Form 8k,Pharmaceuticals,Zoetis Inc.,2024-05-23,0
23845,Declaration of Voting Results by Zoetis Inc,Other,Pharmaceuticals,Zoetis Inc.,2024-05-23,0
23846,Zoetis Inc. Presents at Stifel Jaws & Paws Con...,Business Wire; Company Website,Pharmaceuticals,Zoetis Inc.,2024-05-29,0
23847,Zoetis Inc. Presents at The 44th Annual Willia...,PR Newswire; Business Wire; GlobeNewswire; Com...,Pharmaceuticals,Zoetis Inc.,2024-06-04,1


In [12]:
test_df.to_csv("./Data/test_labelled.csv", index=False)